In [1]:
import random

import numpy as np

In [2]:
class Allocator:
    def __init__(self, arena_size, obj_size, pointer_type):
        self.obj_size = obj_size

        self.arena = np.zeros(arena_size, dtype=pointer_type)
        self.head = -1
        self.bump = 0

    def __repr__(self):
        return f"<Allocator {self.head} {self.bump}: {str(self.arena)}>"

    def mark(self, at, value):
        self.arena[at : at + self.obj_size] = value
        return at

    def malloc(self, mark):
        if self.head == -1:
            if self.bump == len(self.arena):
                raise Exception("cannot allocate")
            else:
                out = self.bump
                self.bump += self.obj_size
                return self.mark(out, mark)
        else:
            out = self.head
            self.head = self.arena[self.head]
            return self.mark(out, mark)
    
    def free(self, at):
        self.arena[at] = self.head
        self.head = at

In [3]:
allocator = Allocator(16, 2, np.int32)
print(allocator)

hold = []

for i in range(6):
    at = allocator.malloc(999)
    print(at)
    print(allocator)
    hold.append(at)

while True:
    action = random.choice(["malloc", "malloc", "free", "free", "free"])
    if action == "malloc":
        at = allocator.malloc(999)
        print("malloc ->", at)
        print(allocator)
        hold.append(at)
    else:
        if len(hold) == 0:
            raise Exception("nothing to free")
        at = random.choice(hold)
        print("free", at)
        allocator.free(at)
        print(allocator)
        hold.remove(at)

<Allocator -1 0: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]>
0
<Allocator -1 2: [999 999   0   0   0   0   0   0   0   0   0   0   0   0   0   0]>
2
<Allocator -1 4: [999 999 999 999   0   0   0   0   0   0   0   0   0   0   0   0]>
4
<Allocator -1 6: [999 999 999 999 999 999   0   0   0   0   0   0   0   0   0   0]>
6
<Allocator -1 8: [999 999 999 999 999 999 999 999   0   0   0   0   0   0   0   0]>
8
<Allocator -1 10: [999 999 999 999 999 999 999 999 999 999   0   0   0   0   0   0]>
10
<Allocator -1 12: [999 999 999 999 999 999 999 999 999 999 999 999   0   0   0   0]>
free 8
<Allocator 8 12: [999 999 999 999 999 999 999 999  -1 999 999 999   0   0   0   0]>
malloc -> 8
<Allocator -1 12: [999 999 999 999 999 999 999 999 999 999 999 999   0   0   0   0]>
free 10
<Allocator 10 12: [999 999 999 999 999 999 999 999 999 999  -1 999   0   0   0   0]>
malloc -> 10
<Allocator -1 12: [999 999 999 999 999 999 999 999 999 999 999 999   0   0   0   0]>
malloc -> 12
<Allocator -1 14: [999 999 999 999 99

Exception: cannot allocate

In [4]:
def next_power_2(v, pointer_type):
    # https://graphics.stanford.edu/~seander/bithacks.html#RoundUpPowerOf2
    if np.dtype(pointer_type) == np.dtype(np.uint32):
        v -= 1
        v |= v >> 1
        v |= v >> 2
        v |= v >> 4
        v |= v >> 8
        v |= v >> 16
        v += 1
        return v
    else:
        raise NotImplementedError

In [5]:
def node(l, r, data):
    if l >= r:
        return
    print("(", sep="", end="")
    if l + 1 == r:
        print(data[l], " ", 999, sep="", end="")
    elif l + 2 == r:
        print(data[l], " ", data[l + 1], sep="", end="")
    else:
        step = next_power_2((r - l) >> 1, np.uint32)
        node(l, l + step, data)
        print(" ", sep="", end="")
        node(l + step, r, data)
    print(")", end="")

def start(data):
    node(0, len(data), data)

start(np.arange(17))

((((0 1) (2 3)) ((4 5) (6 7))) (((8 9) (10 11)) ((12 13) ((14 999) (15 16)))))

In [6]:
pointer_type = np.dtype(np.uint32)

class Heap:
    END_OF_FREE = pointer_type.type(-1)
    LABEL_NIL = pointer_type.type(0)
    LABEL_TYPE = pointer_type.type(1)
    LABEL_INTEGER = pointer_type.type(2)
    LABEL_RECORD = pointer_type.type(3)
    LABEL_RECORD_NODE = pointer_type.type(4)

    NAMES = ["a", "b", "c", "x", "y", "z"]
    TYPES = [(3, 4, 5), (3, 0), (0, 1, 2, 3, 5)]

    def __init__(self, arena_size):
        assert arena_size % 2 == 0
        self.refcount = np.zeros(arena_size >> 1, dtype=np.uint8)
        self.arena = np.zeros(arena_size, dtype=pointer_type)
        self.head = self.END_OF_FREE
        self.bump = 0

    def __repr__(self):
        with np.printoptions(formatter={"int": hex}):
            tmp = str(self.arena).replace("\n", "")
        return f"<Heap {self.head} {self.bump}: {str(self.refcount)} {tmp}>"

    def alloc_cons(self):
        if self.head == self.END_OF_FREE:
            if self.bump == len(self.arena):
                raise Exception("cannot allocate cons")
            else:
                out = self.bump
                self.bump += 2   # next cons
                return out, out + 1
        else:
            out = self.head
            self.head = self.arena[self.head]
            return out, out + 1

    def free_cons(self, car):
        self.arena[car] = self.head
        self.arena[car + 1] = self.END_OF_FREE
        self.head = car

    def labeled(self, label, value):
        shift = (pointer_type.itemsize - 1) * 8   # constant
        return value | (np.uint8(label) << shift)

    def unlabeled(self, labeled_value):
        shift = (pointer_type.itemsize - 1) * 8   # constant
        label_mask = 0xff << shift   # also constant
        label = (labeled_value & label_mask) >> shift
        value = labeled_value & ~label_mask
        return label, value

    def _new_object_node(self, l, r, values):
        if l >= r:
            return -1
        car, cdr = self.alloc_cons()
        if l + 1 == r:
            self.arena[car] = values[l]
            self.arena[cdr] = self.LABEL_NIL
        elif l + 2 == r:
            self.arena[car] = values[l]
            self.arena[cdr] = values[l + 1]
        else:
            step = next_power_2((r - l) >> 1, self.arena.dtype)
            self.arena[car] = self._new_object_node(l, l + step, values)
            self.arena[cdr] = self._new_object_node(l + step, r, values)
        self.refcount[car >> 1] = 1
        return self.labeled(self.LABEL_RECORD_NODE, car)

    def new_object(self, typeid, values):
        nameids = self.TYPES[typeid]
        assert len(nameids) == len(values)
        car, cdr = self.alloc_cons()
        self.arena[car] = self.labeled(self.LABEL_TYPE, typeid)
        self.arena[cdr] = self._new_object_node(0, len(values), values)
        self.refcount[car >> 1] = 1
        return self.labeled(self.LABEL_RECORD, car)
    
    def del_object(self, labeled_value):
        label, value = self.unlabeled(labeled_value)
        if labeled_value == self.LABEL_NIL:
            pass
        elif label == self.LABEL_INTEGER:
            pass
        elif label == self.LABEL_RECORD:
            self.refcount[value >> 1] -= 1
            if self.refcount[value >> 1] == 0:
                self.del_object(self.arena[value + 1])
                self.free_cons(value)
        elif label == self.LABEL_RECORD_NODE:
            self.refcount[value >> 1] -= 1
            if self.refcount[value >> 1] == 0:
                self.del_object(self.arena[value])
                self.del_object(self.arena[value + 1])
                self.free_cons(value)
                
    def _view_object_values(self, l, r, labeled_value):
        label, car = self.unlabeled(labeled_value)
        cdr = car + 1
        if l >= r:
            pass
        elif l + 1 == r:
            yield self.view_object(self.arena[car])
        elif l + 2 == r:
            yield self.view_object(self.arena[car])
            yield self.view_object(self.arena[cdr])
        else:
            step = next_power_2((r - l) >> 1, self.arena.dtype)
            yield from self._view_object_values(l, l + step, self.arena[car])
            yield from self._view_object_values(l + step, r, self.arena[cdr])

    def view_object(self, labeled_value):
        label, value = self.unlabeled(labeled_value)
        if label == self.LABEL_INTEGER:
            return value
        elif label == self.LABEL_RECORD:
            _, typeid = self.unlabeled(self.arena[value])
            fields = [self.NAMES[stringid] for stringid in self.TYPES[typeid]]
            return dict(zip(fields, self._view_object_values(0, len(fields), self.arena[value + 1])))
        else:
            raise Exception("no value here")

In [7]:
heap = Heap(18)
heap

<Heap 4294967295 0: [0 0 0 0 0 0 0 0 0] [0x0 0x0 0x0 0x0 0x0 0x0 0x0 0x0 0x0 0x0 0x0 0x0 0x0 0x0 0x0 0x0 0x0 0x0]>

In [8]:
one = heap.new_object(0, [heap.labeled(heap.LABEL_INTEGER, x) for x in [8, 9, 10]])

In [9]:
heap

<Heap 4294967295 8: [1 1 1 1 0 0 0 0 0] [0x1000000 0x4000002 0x4000004 0x4000006 0x2000008 0x0 0x2000009 0x200000a 0x0 0x0 0x0 0x0 0x0 0x0 0x0 0x0 0x0 0x0]>

In [10]:
heap.view_object(one)

{'x': 8, 'y': 9, 'z': 10}

In [11]:
two = heap.new_object(1, [heap.labeled(heap.LABEL_INTEGER, x) for x in [5, 6]])

In [12]:
heap

<Heap 4294967295 12: [1 1 1 1 1 1 0 0 0] [0x1000000 0x4000002 0x4000004 0x4000006 0x2000008 0x0 0x2000009 0x200000a 0x1000001 0x400000a 0x2000005 0x2000006 0x0 0x0 0x0 0x0 0x0 0x0]>

In [13]:
heap.view_object(two)

{'x': 5, 'a': 6}

In [14]:
heap.del_object(one)

In [15]:
heap

<Heap 0 12: [0 0 0 0 1 1 0 0 0] [0x2 0xffffffff 0x6 0xffffffff 0xffffffff 0xffffffff 0x4 0xffffffff 0x1000001 0x400000a 0x2000005 0x2000006 0x0 0x0 0x0 0x0 0x0 0x0]>

In [16]:
three = heap.new_object(1, [heap.labeled(heap.LABEL_INTEGER, x) for x in [7, 5]])

In [17]:
heap

<Heap 6 12: [1 1 0 0 1 1 0 0 0] [0x1000001 0x4000002 0x2000007 0x2000005 0xffffffff 0xffffffff 0x4 0xffffffff 0x1000001 0x400000a 0x2000005 0x2000006 0x0 0x0 0x0 0x0 0x0 0x0]>

In [18]:
heap.view_object(three)

{'x': 7, 'a': 5}

In [19]:
four = heap.new_object(0, [heap.labeled(heap.LABEL_INTEGER, x) for x in [1, 2, 3]])

In [20]:
heap

<Heap 4294967295 16: [1 1 1 1 1 1 1 1 0] [0x1000001 0x4000002 0x2000007 0x2000005 0x400000c 0x400000e 0x1000000 0x4000004 0x1000001 0x400000a 0x2000005 0x2000006 0x2000001 0x0 0x2000002 0x2000003 0x0 0x0]>

In [21]:
heap.view_object(four)

{'x': 1, 'y': 2, 'z': 3}

In [22]:
heap.new_object(1, [heap.labeled(heap.LABEL_INTEGER, x) for x in [7, 5]])

Exception: cannot allocate cons